In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
datapath = "data/"
raw_beach_weather_stations = pd.read_csv(datapath + "Beach_Weather_Stations_-_Automated_Sensors.csv")
print(raw_beach_weather_stations['Station Name'].value_counts().sort_index())
# station_counts = defaultdict(int)
# for station in raw_beach_weather_stations['Station Name']:
#     station_counts[station] += 1
# print(station_counts)


Station Name
63rd Street Weather Station    49951
Foster Weather Station         79174
Oak Street Weather Station     73573
Name: count, dtype: int64


/var/folders/9g/gwn33xx13pd4b2km54vx15qw0000gn/T/ipykernel_5222/2456480371.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_beach_weather_stations = pd.read_csv(datapath + "Beach_Weather_Stations_-_Automated_Sensors.csv")


In [3]:
station_location = pd.read_csv(datapath + "Beach_Water_and_Weather_Sensor_Locations_20260423.csv")
station_location

,Sensor Name,Sensor Type,Latitude,Longitude,Location
0,63rd Street Beach,Water,41.784561,-87.571453,"(41.784561, -87.571453)"
1,Calumet Beach,Water,41.714739,-87.527356,"(41.714739, -87.527356)"
2,Montrose Beach,Water,41.969094,-87.638003,"(41.969094, -87.638003)"
3,Ohio Street Beach,Water,41.894328,-87.613083,"(41.894328, -87.613083)"
4,Osterman Beach,Water,41.987675,-87.651008,"(41.987675, -87.651008)"
5,Rainbow Beach,Water,41.760147,-87.550081,"(41.760147, -87.550081)"
6,63rd Street Weather Station,Weather,41.780992,-87.572619,"(41.780992, -87.572619)"
7,Foster Weather Station,Weather,41.976464,-87.647525,"(41.976464, -87.647525)"
8,Oak Street Weather Station,Weather,41.901997,-87.622817,"(41.901997, -87.622817)"


### Quickly checking water sensor data

compare ohio street bouy with the NOAA bouy wave height. expect wave height to be lower.

john meeting notes - #convert to float
#can use libraries that better report on a dataset now called y data used to be panda profiling

In [4]:
#here are the water stations (reading of actually in the water, pretty messy data from first glance
#maybe only use wave height and wave period, but would need thorough checking - I think these sensors are less reliable than weather
raw_beach_water_stations = pd.read_csv(datapath + "Beach_Water_Quality_-_Automated_Sensors.csv")
display(raw_beach_water_stations)

print(f"Wave Height summary {raw_beach_water_stations['Wave Height'].describe()}")
print("~~~~")
print(f"Wave Period summary {raw_beach_water_stations['Wave Period'].describe()}")
# display(raw_beach_water_stations)
print("~~~~")
raw_beach_water_stations['year'] = pd.to_datetime(raw_beach_water_stations['Measurement Timestamp'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce').dt.year
print(f"total readings per year {raw_beach_water_stations['year'].value_counts().sort_index()}")
#this is getting phased out, they don't seem to use them consistently....

,Beach Name,Measurement Timestamp,Water Temperature,Turbidity,Transducer Depth,Wave Height,Wave Period,Battery Life,Measurement Timestamp Label,Measurement ID
0,Ohio Street Beach,07/07/2025 08:00:00 PM,"-100,000","-100,000",NaN,0,6.3,14.1,07/07/2025 8:00 PM,OhioStreetBeach202507072000
1,Ohio Street Beach,07/07/2025 07:00:00 PM,"-100,000","-100,000",NaN,0,6.5,14.5,07/07/2025 7:00 PM,OhioStreetBeach202507071900
2,Ohio Street Beach,07/07/2025 06:00:00 PM,"-100,000","-100,000",NaN,0,6.5,14.5,07/07/2025 6:00 PM,OhioStreetBeach202507071800
3,Ohio Street Beach,07/07/2025 05:00:00 PM,"-100,000","-100,000",NaN,0,6.6,14.3,07/07/2025 5:00 PM,OhioStreetBeach202507071700
4,Ohio Street Beach,07/07/2025 04:00:00 PM,"-100,000","-100,000",NaN,0,6.5,13.8,07/07/2025 4:00 PM,OhioStreetBeach202507071600
...,...,...,...,...,...,...,...,...,...,...
46948,63rd Street Beach,09/18/2013 10:00:00 AM,18.9,7.56,1.517,0.14,4,11.0,09/18/2013 10:00 AM,63rdStreetBeach201309181000
46949,Calumet Beach,09/03/2013 04:00:00 PM,23.2,3.63,1.201,0.174,6,9.4,9/3/2013 4:00 PM,CalumetBeach201309031600
46950,Ohio Street Beach,09/03/2013 03:00:00 AM,21.9,4.97,1.039,0.241,7,9.4,09/03/2013 3:00 AM,OhioStreetBeach201309030300
46951,Osterman Beach,08/31/2013 11:00:00 PM,21.5,3.51,1.538,0.231,4,9.4,08/31/2013 11:00 PM,OstermanBeach201308312300


Wave Height summary count     46340
unique      630
top       0.187
freq       2835
Name: Wave Height, dtype: object
~~~~
Wave Period summary count     46340
unique       72
top           3
freq      16312
Name: Wave Period, dtype: object
~~~~
total readings per year year
2013        5
2014    10256
2015    14341
2016     7601
2017     2714
2018     1949
2019     2603
2020      304
2021      935
2023     1863
2024     3285
2025     1097
Name: count, dtype: int64


## Using weather station data which is land based data
It looks like they are phasing out the water based sensors, but we might be able to supplement water readings from NAOO bouys

In [5]:
raw_beach_weather_stations

,Station Name,Measurement Timestamp,Air Temperature,Wet Bulb Temperature,Humidity,Rain Intensity,Interval Rain,Total Rain,Precipitation Type,Wind Direction,Wind Speed,Maximum Wind Speed,Barometric Pressure,Solar Radiation,Heading,Battery Life,Measurement Timestamp Label,Measurement ID
0,Foster Weather Station,04/23/2026 11:00:00 AM,16.50,NaN,75,NaN,0.0,NaN,NaN,0,3.3,0.0,989.8,685,NaN,15.1,04/23/2026 11:00 AM,FosterWeatherStation202604231100
1,Oak Street Weather Station,04/23/2026 11:00:00 AM,21.00,16.9,67,0.0,0.0,101.8,0.0,110,4.1,5.1,991.5,689,358.0,12.0,04/23/2026 11:00 AM,OakStreetWeatherStation202604231100
2,Foster Weather Station,04/23/2026 10:00:00 AM,16.28,NaN,75,NaN,0.0,NaN,NaN,0,3.3,0.0,990.2,568,NaN,15.1,04/23/2026 10:00 AM,FosterWeatherStation202604231000
3,Oak Street Weather Station,04/23/2026 10:00:00 AM,21.40,17.0,64,0.0,0.0,101.8,0.0,121,2.4,3.5,991.9,590,358.0,12.0,04/23/2026 10:00 AM,OakStreetWeatherStation202604231000
4,Foster Weather Station,04/23/2026 09:00:00 AM,19.67,NaN,65,NaN,0.0,NaN,NaN,0,3.3,0.0,990.5,423,NaN,15.1,04/23/2026 9:00 AM,FosterWeatherStation202604230900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202693,Oak Street Weather Station,05/22/2015 05:00:00 PM,NaN,6.3,56,0.0,0.0,1.4,0.0,124,1.5,2.3,NaN,180,322.0,12.1,05/22/2015 5:00 PM,OakStreetWeatherStation201505221700
202694,Foster Weather Station,05/22/2015 04:00:00 PM,9.17,NaN,59,NaN,0.0,NaN,NaN,4,4.0,4.4,NaN,556,NaN,15.1,05/22/2015 4:00 PM,FosterWeatherStation201505221600
202695,Oak Street Weather Station,05/22/2015 03:00:00 PM,NaN,7.0,55,0.0,0.0,1.4,0.0,63,1.9,2.8,NaN,780,322.0,12.0,05/22/2015 3:00 PM,OakStreetWeatherStation201505221500
202696,63rd Street Weather Station,04/30/2015 05:00:00 AM,6.10,4.3,76,0.0,0.0,2.5,0.0,11,7.2,13.0,989.9,4,354.0,11.9,04/30/2015 5:00 AM,63rdStreetWeatherStation201504300500


In [6]:
print("Shape:", raw_beach_weather_stations.shape)
print("\nColumn types:\n", raw_beach_weather_stations.dtypes)

# NaN counts
nan_counts = raw_beach_weather_stations.isna().sum()
print("\nNaN counts:\n", nan_counts[nan_counts > 0])

# Numeric column stats — spot outliers via min/max
print("\nNumeric summary:")
display(raw_beach_weather_stations.describe())

# Duplicates
dupes = raw_beach_weather_stations.duplicated().sum()
print(f"\nDuplicate rows: {dupes}")

# Categorical columns — check cardinality and spot unexpected values
cat_cols = raw_beach_weather_stations.select_dtypes(include='object').columns
for col in cat_cols:
    n = raw_beach_weather_stations[col].nunique()
    print(f"\n{col} ({n} unique):")
    print(raw_beach_weather_stations[col].value_counts().head(10))


Shape: (202698, 18)

Column types:
 Station Name                    object
Measurement Timestamp           object
Air Temperature                float64
Wet Bulb Temperature           float64
Humidity                         int64
Rain Intensity                 float64
Interval Rain                  float64
Total Rain                      object
Precipitation Type             float64
Wind Direction                   int64
Wind Speed                     float64
Maximum Wind Speed             float64
Barometric Pressure            float64
Solar Radiation                 object
Heading                        float64
Battery Life                   float64
Measurement Timestamp Label     object
Measurement ID                  object
dtype: object

NaN counts:
 Air Temperature            75
Wet Bulb Temperature    79174
Rain Intensity          79174
Total Rain              79174
Precipitation Type      79174
Barometric Pressure       146
Heading                 79174
dtype: int64

Numeric su

,Air Temperature,Wet Bulb Temperature,Humidity,Rain Intensity,Interval Rain,Precipitation Type,Wind Direction,Wind Speed,Maximum Wind Speed,Barometric Pressure,Heading,Battery Life
count,202623.000000,123524.000000,202698.000000,123524.000000,202698.000000,123524.000000,202698.000000,202698.000000,202698.000000,202552.000000,123524.000000,202698.000000
mean,12.300787,10.027720,67.895376,0.156884,0.141568,4.275566,139.446438,2.920852,3.518447,994.328469,283.897291,13.174987
std,10.542579,9.492819,15.612178,1.776049,1.094257,15.610866,122.586512,5.708207,6.287727,9.990938,141.437930,1.546753
min,-29.780000,-28.900000,0.000000,0.000000,-0.900000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.000000,2.700000,57.000000,0.000000,0.000000,0.000000,8.000000,1.600000,1.100000,990.200000,350.000000,11.900000
50%,13.170000,11.100000,69.000000,0.000000,0.000000,0.000000,113.000000,3.000000,3.000000,994.400000,355.000000,12.000000
75%,21.300000,18.300000,80.000000,0.000000,0.000000,0.000000,260.000000,3.300000,5.200000,998.700000,357.000000,15.100000
max,37.600000,28.400000,100.000000,183.600000,63.420000,70.000000,359.000000,999.900000,999.900000,3098.500000,359.000000,15.300000



Duplicate rows: 0

Station Name (3 unique):
Station Name
Foster Weather Station         79174
Oak Street Weather Station     73573
63rd Street Weather Station    49951
Name: count, dtype: int64

Measurement Timestamp (84804 unique):
Measurement Timestamp
06/22/2015 09:00:00 AM    3
06/22/2015 10:00:00 AM    3
06/22/2015 11:00:00 AM    3
06/22/2015 12:00:00 PM    3
06/22/2015 01:00:00 PM    3
06/22/2015 03:00:00 PM    3
06/22/2015 04:00:00 PM    3
06/22/2015 05:00:00 PM    3
06/22/2015 06:00:00 PM    3
06/22/2015 07:00:00 PM    3
Name: count, dtype: int64

Total Rain (4218 unique):
Total Rain
0.0      1443
3.2       939
13.0      797
1.4       551
11.8      466
15.3      465
276.7     463
25.3      446
1.2       428
8.7       414
Name: count, dtype: int64

Solar Radiation (1077 unique):
Solar Radiation
0     46728
4     12066
1     11053
2     10998
3      9287
5      7022
6      4623
-3     4363
-4     3690
-2     3402
Name: count, dtype: int64

Measurement Timestamp Label (84804 uniq

In [7]:
numeric_cols = raw_beach_weather_stations.select_dtypes(include='number').columns.tolist()
station_col = 'Station Name'

# NaN rate per station
nan_by_station = (raw_beach_weather_stations
    .groupby(station_col)[numeric_cols]
    .apply(lambda x: x.isna().mean() * 100)
    .round(1))
print("% NaN per station per column:")
display(nan_by_station)

# Stats per station for each numeric column
for col in numeric_cols:
    print(f"\n--- {col} ---")
    display(raw_beach_weather_stations.groupby(station_col)[col].agg(['min', 'mean', 'max', 'std']).round(2))


% NaN per station per column:


,Air Temperature,Wet Bulb Temperature,Humidity,Rain Intensity,Interval Rain,Precipitation Type,Wind Direction,Wind Speed,Maximum Wind Speed,Barometric Pressure,Heading,Battery Life
Station Name,,,,,,,,,,,,
63rd Street Weather Station,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Foster Weather Station,0.0,100.0,0.0,100.0,0.0,100.0,0.0,0.0,0.0,0.1,100.0,0.0
Oak Street Weather Station,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.0,0.0



--- Air Temperature ---


,min,mean,max,std
Station Name,,,,
63rd Street Weather Station,-28.50,13.22,36.40,10.52
Foster Weather Station,-29.78,11.12,37.28,10.54
Oak Street Weather Station,-21.40,12.95,37.60,10.44



--- Wet Bulb Temperature ---


...

,min,mean,max,std
Station Name,,,,
63rd Street Weather Station,0.0,994.18,1022.7,12.80
Foster Weather Station,958.7,993.75,3098.5,10.29
Oak Street Weather Station,960.1,995.05,1019.6,7.00



--- Heading ---


,min,mean,max,std
Station Name,,,,
63rd Street Weather Station,0.0,353.05,358.0,10.89
Foster Weather Station,NaN,NaN,NaN,NaN
Oak Street Weather Station,0.0,236.95,359.0,167.50



--- Battery Life ---


,min,mean,max,std
Station Name,,,,
63rd Street Weather Station,0.0,11.88,14.1,0.15
Foster Weather Station,11.9,15.10,15.3,0.11
Oak Street Weather Station,11.2,11.98,12.2,0.09


In [8]:
weather_df = raw_beach_weather_stations.copy()

# 999.9 reading is probably a sentinel error value in wind columns mark them as nan
wind_cols = ['Wind Speed', 'Maximum Wind Speed']
for col in wind_cols:
    weather_df[col] = weather_df[col].replace(999.9, np.nan)

#narometric pressure: 0.0 is sensor offline and >1100 is physically impossible, mark them as nan
weather_df['Barometric Pressure'] = weather_df['Barometric Pressure'].where(
    weather_df['Barometric Pressure'].between(850, 1100), np.nan
)

#negative interval rain is impossible, just clip it but remember that this might be a miscalibrated sensor
weather_df['Interval Rain'] = weather_df['Interval Rain'].clip(lower=0)

#verify fixes
print("Remaining 999.9 in wind cols:", (weather_df[wind_cols] == 999.9).sum().sum())
print("Barometric pressure range:")
print(weather_df.groupby('Station Name')['Barometric Pressure'].agg(['min', 'max']))
print("\nInterval Rain min:", weather_df['Interval Rain'].min())

weather_df


Remaining 999.9 in wind cols: 0
Barometric pressure range:
                               min     max
Station Name                              
63rd Street Weather Station  964.7  1022.7
Foster Weather Station       958.7  1022.4
Oak Street Weather Station   960.1  1019.6

Interval Rain min: 0.0


,Station Name,Measurement Timestamp,Air Temperature,Wet Bulb Temperature,Humidity,Rain Intensity,Interval Rain,Total Rain,Precipitation Type,Wind Direction,Wind Speed,Maximum Wind Speed,Barometric Pressure,Solar Radiation,Heading,Battery Life,Measurement Timestamp Label,Measurement ID
0,Foster Weather Station,04/23/2026 11:00:00 AM,16.50,NaN,75,NaN,0.0,NaN,NaN,0,3.3,0.0,989.8,685,NaN,15.1,04/23/2026 11:00 AM,FosterWeatherStation202604231100
1,Oak Street Weather Station,04/23/2026 11:00:00 AM,21.00,16.9,67,0.0,0.0,101.8,0.0,110,4.1,5.1,991.5,689,358.0,12.0,04/23/2026 11:00 AM,OakStreetWeatherStation202604231100
2,Foster Weather Station,04/23/2026 10:00:00 AM,16.28,NaN,75,NaN,0.0,NaN,NaN,0,3.3,0.0,990.2,568,NaN,15.1,04/23/2026 10:00 AM,FosterWeatherStation202604231000
3,Oak Street Weather Station,04/23/2026 10:00:00 AM,21.40,17.0,64,0.0,0.0,101.8,0.0,121,2.4,3.5,991.9,590,358.0,12.0,04/23/2026 10:00 AM,OakStreetWeatherStation202604231000
4,Foster Weather Station,04/23/2026 09:00:00 AM,19.67,NaN,65,NaN,0.0,NaN,NaN,0,3.3,0.0,990.5,423,NaN,15.1,04/23/2026 9:00 AM,FosterWeatherStation202604230900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202693,Oak Street Weather Station,05/22/2015 05:00:00 PM,NaN,6.3,56,0.0,0.0,1.4,0.0,124,1.5,2.3,NaN,180,322.0,12.1,05/22/2015 5:00 PM,OakStreetWeatherStation201505221700
202694,Foster Weather Station,05/22/2015 04:00:00 PM,9.17,NaN,59,NaN,0.0,NaN,NaN,4,4.0,4.4,NaN,556,NaN,15.1,05/22/2015 4:00 PM,FosterWeatherStation201505221600
202695,Oak Street Weather Station,05/22/2015 03:00:00 PM,NaN,7.0,55,0.0,0.0,1.4,0.0,63,1.9,2.8,NaN,780,322.0,12.0,05/22/2015 3:00 PM,OakStreetWeatherStation201505221500
202696,63rd Street Weather Station,04/30/2015 05:00:00 AM,6.10,4.3,76,0.0,0.0,2.5,0.0,11,7.2,13.0,989.9,4,354.0,11.9,04/30/2015 5:00 AM,63rdStreetWeatherStation201504300500


Foster is very different, has less sensors than Oak St and 63rd street. It is closest to the beach I like though. Leave this for now, see how data might overlap with other weather sources before fully cleaning.

I'm not sure if I should omit interval rain data, all other rain data isn't sensed so how could interval be sensed? 6717 of 79174 rows have a value > 0.0 for rain interval

john - this data on these dates to the other stations interval rain. interval rain how it's sensed and what it is. if it doesn't line up then get rid of it if it does can calc it

Notes for joining other datasets, city of chicago data are assumed to be in Central Time, the timestamps are floating and can be assumed to be the exact time that happened on that day. https://dev.socrata.com/docs/datatypes/floating_timestamp.html#,

In [9]:
foster_weather_df = duckdb.query("""
SELECT
    *
    FROM weather_df 
    WHERE "Station Name" = 'Foster Weather Station';                                       
""").df()
# print(foster_weather_df['Total Rain'].isna().sum()) #all total rain is not there too
print("~~~foster stats~~~")
print(foster_weather_df['Interval Rain'].describe())
print((foster_weather_df['Interval Rain'] != 0.0).sum())
rain_reading = foster_weather_df['Interval Rain'] != 0.0
# display(foster_weather_df)
# foster_weather_df = foster_weather_df.drop(columns=['Wet Bulb Temperature', 'Rain Intensity', 'Interval Rain', 'Total Rain','Precipitation Type', 'Heading'])
# display(foster_weather_df)

oakst_weather_df = duckdb.query("""
SELECT
    *
    FROM weather_df 
    WHERE "Station Name" = 'Oak Street Weather Station';                                       
""").df()
# print(foster_weather_df['Total Rain'].isna().sum()) #all total rain is not there too
print("~~~oak street stats~~~")
print(oakst_weather_df['Interval Rain'].describe())
print((oakst_weather_df['Interval Rain'] != 0.0).sum())
sixtythirdst_weather_df = duckdb.query("""
SELECT
    *
    FROM weather_df 
    WHERE "Station Name" = '63rd Street Weather Station';                                       
""").df()
print("~~~63rd street stats~~~")
print(sixtythirdst_weather_df['Interval Rain'].describe())
print((sixtythirdst_weather_df['Interval Rain'] != 0.0).sum())


~~~foster stats~~~
count    79174.000000
mean         0.128169
std          1.133298
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         63.420000
Name: Interval Rain, dtype: float64
6717
~~~oak street stats~~~
count    73573.000000
mean         0.102142
std          0.620369
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         16.500000
Name: Interval Rain, dtype: float64
5574
~~~63rd street stats~~~
count    49951.000000
mean         0.220894
std          1.499173
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         50.400000
Name: Interval Rain, dtype: float64
4037


the rain interval stats look similar between the stations. we may keep them, can calc the interval rain to be precipitation sum and the other ones. should def be checking the other stations and doing the same calcs to see if my calcs are correct

the interval rain does look good. it's description is: Interval Rain : Rain since the last hourly measurement, in mm63rd street doesn;t have nearly as many values though.

from https://data.cityofchicago.org/Parks-Recreation/Beach-Weather-Stations-Automated-Sensors/k7hf-8y75/about_data

In [10]:
sixtythirdst_weather_df.dtypes

Station Name                    object
Measurement Timestamp           object
Air Temperature                float64
Wet Bulb Temperature           float64
Humidity                         int64
Rain Intensity                 float64
Interval Rain                  float64
Total Rain                      object
Precipitation Type             float64
Wind Direction                   int64
Wind Speed                     float64
Maximum Wind Speed             float64
Barometric Pressure            float64
Solar Radiation                 object
Heading                        float64
Battery Life                   float64
Measurement Timestamp Label     object
Measurement ID                  object
dtype: object

In [11]:
# 50 or 63 mm in an hour does not seem feasible, max recorded is 305 mm
dataframes = [foster_weather_df,oakst_weather_df, sixtythirdst_weather_df]
# for df in dataframes:
#     print(df["Station Name"].unique)
#     sns.boxplot(data=df[['Interval Rain']])
#     plt.title('Boxplot')
#     plt.show()
weather_df

,Station Name,Measurement Timestamp,Air Temperature,Wet Bulb Temperature,Humidity,Rain Intensity,Interval Rain,Total Rain,Precipitation Type,Wind Direction,Wind Speed,Maximum Wind Speed,Barometric Pressure,Solar Radiation,Heading,Battery Life,Measurement Timestamp Label,Measurement ID
0,Foster Weather Station,04/23/2026 11:00:00 AM,16.50,NaN,75,NaN,0.0,NaN,NaN,0,3.3,0.0,989.8,685,NaN,15.1,04/23/2026 11:00 AM,FosterWeatherStation202604231100
1,Oak Street Weather Station,04/23/2026 11:00:00 AM,21.00,16.9,67,0.0,0.0,101.8,0.0,110,4.1,5.1,991.5,689,358.0,12.0,04/23/2026 11:00 AM,OakStreetWeatherStation202604231100
2,Foster Weather Station,04/23/2026 10:00:00 AM,16.28,NaN,75,NaN,0.0,NaN,NaN,0,3.3,0.0,990.2,568,NaN,15.1,04/23/2026 10:00 AM,FosterWeatherStation202604231000
3,Oak Street Weather Station,04/23/2026 10:00:00 AM,21.40,17.0,64,0.0,0.0,101.8,0.0,121,2.4,3.5,991.9,590,358.0,12.0,04/23/2026 10:00 AM,OakStreetWeatherStation202604231000
4,Foster Weather Station,04/23/2026 09:00:00 AM,19.67,NaN,65,NaN,0.0,NaN,NaN,0,3.3,0.0,990.5,423,NaN,15.1,04/23/2026 9:00 AM,FosterWeatherStation202604230900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202693,Oak Street Weather Station,05/22/2015 05:00:00 PM,NaN,6.3,56,0.0,0.0,1.4,0.0,124,1.5,2.3,NaN,180,322.0,12.1,05/22/2015 5:00 PM,OakStreetWeatherStation201505221700
202694,Foster Weather Station,05/22/2015 04:00:00 PM,9.17,NaN,59,NaN,0.0,NaN,NaN,4,4.0,4.4,NaN,556,NaN,15.1,05/22/2015 4:00 PM,FosterWeatherStation201505221600
202695,Oak Street Weather Station,05/22/2015 03:00:00 PM,NaN,7.0,55,0.0,0.0,1.4,0.0,63,1.9,2.8,NaN,780,322.0,12.0,05/22/2015 3:00 PM,OakStreetWeatherStation201505221500
202696,63rd Street Weather Station,04/30/2015 05:00:00 AM,6.10,4.3,76,0.0,0.0,2.5,0.0,11,7.2,13.0,989.9,4,354.0,11.9,04/30/2015 5:00 AM,63rdStreetWeatherStation201504300500


# Preparation for merging and comparing

In [12]:
#preparation for 
#making a functional timestamp:
weather_df['datetime_central_time'] = (
    pd.to_datetime(
        weather_df['Measurement Timestamp'],
        format='%m/%d/%Y %I:%M:%S %p',
        errors='coerce'
    )
    .dt.tz_localize('America/Chicago', nonexistent='NaT', ambiguous='NaT')) #nonexistent='shift_forward' handles the spring-forward case (the hour that doesn't exist gets bumped up). For fall-back, ambiguous='infer' works if your data is monotonic; 
weather_df
#
#making a functional df_for eda
weather_df= weather_df.rename(columns = {'Interval Rain':'COC_mmprecip_lag1h', 'Station Name' : 'station_name', 'datetime_central_time':'timestamp_central'})
beach_coc_weather_precip_hourly = weather_df[['station_name', 'COC_mmprecip_lag1h','timestamp_central']]
beach_coc_weather_precip_hourly['timestamp_central']



0        2026-04-23 11:00:00-05:00
1        2026-04-23 11:00:00-05:00
2        2026-04-23 10:00:00-05:00
3        2026-04-23 10:00:00-05:00
4        2026-04-23 09:00:00-05:00
                    ...           
202693   2015-05-22 17:00:00-05:00
202694   2015-05-22 16:00:00-05:00
202695   2015-05-22 15:00:00-05:00
202696   2015-04-30 05:00:00-05:00
202697   2015-04-25 09:00:00-05:00
Name: timestamp_central, Length: 202698, dtype: datetime64[ns, America/Chicago]

In [13]:

sql = """
    SELECT 
        MIN(timestamp_central) as first_obs,
        MAX(timestamp_central) as last_obs,
        COUNT() as count
    FROM beach_coc_weather_precip_hourly
    WHERE "station_name" = '63rd Street Weather Station' AND timestamp_central > '2022-01-01 10:00:00-05:00'
"""
duckdb.sql(sql).df()

,first_obs,last_obs,count
0,2022-01-01 10:00:00-06:00,2025-03-18 00:30:00-05:00,5354


there are 6 readings from 63rd street in 2025, 11 in 2024, 1 in 2023, then like 5k in 2022

In [14]:
station_location = station_location.rename(columns = {'Sensor Name':'station_name'}).copy()

print(station_location.head())
COC_weather_wlatlon = duckdb.query("""
SELECT
    b.station_name, 
    b.COC_mmprecip_lag1h AS coc_mmprecip_sum1h,
    b.timestamp_central,
    s.Latitude AS station_lat,
    s.Longitude AS station_lon,
    FROM beach_coc_weather_precip_hourly b   
    JOIN station_location s ON b.station_name = s.station_name       
""").df()
COC_weather_wlatlon

        station_name Sensor Type   Latitude  Longitude  \
0  63rd Street Beach       Water  41.784561 -87.571453   
1      Calumet Beach       Water  41.714739 -87.527356   
2     Montrose Beach       Water  41.969094 -87.638003   
3  Ohio Street Beach       Water  41.894328 -87.613083   
4     Osterman Beach       Water  41.987675 -87.651008   

                  Location  
0  (41.784561, -87.571453)  
1  (41.714739, -87.527356)  
2  (41.969094, -87.638003)  
3  (41.894328, -87.613083)  
4  (41.987675, -87.651008)  


,station_name,coc_mmprecip_sum1h,timestamp_central,station_lat,station_lon
0,Foster Weather Station,0.0,2026-04-23 11:00:00-05:00,41.976464,-87.647525
1,Oak Street Weather Station,0.0,2026-04-23 11:00:00-05:00,41.901997,-87.622817
2,Foster Weather Station,0.0,2026-04-23 10:00:00-05:00,41.976464,-87.647525
3,Oak Street Weather Station,0.0,2026-04-23 10:00:00-05:00,41.901997,-87.622817
4,Foster Weather Station,0.0,2026-04-23 09:00:00-05:00,41.976464,-87.647525
...,...,...,...,...,...
202693,Oak Street Weather Station,0.0,2015-05-22 17:00:00-05:00,41.901997,-87.622817
202694,Foster Weather Station,0.0,2015-05-22 16:00:00-05:00,41.976464,-87.647525
202695,Oak Street Weather Station,0.0,2015-05-22 15:00:00-05:00,41.901997,-87.622817
202696,63rd Street Weather Station,0.0,2015-04-30 05:00:00-05:00,41.780992,-87.572619


In [15]:
COC_weather_final = COC_weather_wlatlon.copy()
COC_weather_final['station_name'] = COC_weather_final['station_name'].str.replace(" ","_").str.lower().str.replace("_weather_station","").copy()
COC_weather_final 

,station_name,coc_mmprecip_sum1h,timestamp_central,station_lat,station_lon
0,foster,0.0,2026-04-23 11:00:00-05:00,41.976464,-87.647525
1,oak_street,0.0,2026-04-23 11:00:00-05:00,41.901997,-87.622817
2,foster,0.0,2026-04-23 10:00:00-05:00,41.976464,-87.647525
3,oak_street,0.0,2026-04-23 10:00:00-05:00,41.901997,-87.622817
4,foster,0.0,2026-04-23 09:00:00-05:00,41.976464,-87.647525
...,...,...,...,...,...
202693,oak_street,0.0,2015-05-22 17:00:00-05:00,41.901997,-87.622817
202694,foster,0.0,2015-05-22 16:00:00-05:00,41.976464,-87.647525
202695,oak_street,0.0,2015-05-22 15:00:00-05:00,41.901997,-87.622817
202696,63rd_street,0.0,2015-04-30 05:00:00-05:00,41.780992,-87.572619


In [16]:
# commented out so it won't rewrite that data used in modeling so far
# COC_weather_final.to_parquet("data/prepared_data/COC_precip.parquet", index=False)

Need to still configure wind for incorporation

In [17]:
#export to do EDA and use in model
# beach_coc_weather_precip_hourly.to_csv("data/prepared_data/beach_coc_hourly_precip.csv", index=False)

I'm going to implement a feature naming convention now: 
 Feature naming convention for this project:

     {source}_{variable}_{operation}_{window}

 - source:    where the data came from (coc, usgs, beach, season, etc.)

 - variable:  what's being measured (precip, temp, wind, etc.)

 - operation: how it was aggregated (sum, max, mean, count, lag)
 
 - window:    over what time period (24h, 7d, 6h)

 Examples:

   coc_precip_sum_24h     -> sum of COC precip over 24h before sample

   coc_precip_max_6h      -> max single-bucket COC precip in last 6h

   usgs_precip_sum_7d     -> sum of USGS precip over 7 days

   season_sin_doy         -> sin of day-of-year (no source/window needed)

 Rule: if you change ONE thing about a feature, only ONE slot in the
 name should change. This makes filtering by prefix easy.

# Preparing wind features

In [18]:
print('~~~~raw data~~~~')
print(raw_beach_weather_stations.head())
print(raw_beach_weather_stations.shape)
print(raw_beach_weather_stations.dtypes)
print(raw_beach_weather_stations.isna().sum())
print(raw_beach_weather_stations.describe())
print('~~~~prepped data~~~~')
print(weather_df.head())
print(weather_df.shape)
print(weather_df.dtypes)
print(weather_df.isna().sum())
print(weather_df.describe())

~~~~raw data~~~~
                 Station Name   Measurement Timestamp  Air Temperature  \
0      Foster Weather Station  04/23/2026 11:00:00 AM            16.50   
1  Oak Street Weather Station  04/23/2026 11:00:00 AM            21.00   
2      Foster Weather Station  04/23/2026 10:00:00 AM            16.28   
3  Oak Street Weather Station  04/23/2026 10:00:00 AM            21.40   
4      Foster Weather Station  04/23/2026 09:00:00 AM            19.67   

   Wet Bulb Temperature  Humidity  Rain Intensity  Interval Rain Total Rain  \
0                   NaN        75             NaN            0.0        NaN   
1                  16.9        67             0.0            0.0      101.8   
2                   NaN        75             NaN            0.0        NaN   
3                  17.0        64             0.0            0.0      101.8   
4                   NaN        65             NaN            0.0        NaN   

   Precipitation Type  Wind Direction  Wind Speed  Maximum Wind

In [19]:
# 1. Which stations are present, over what dates?
print("Station coverage:")
print(weather_df.groupby('station_name')['timestamp_central'].agg(['min', 'max', 'count']))

# 2. Is Foster's wind direction broken?
print("\nWDIR stats per station:")
print(weather_df.groupby('station_name')['Wind Direction'].agg(['nunique', 'mean', 'std']))

print("\nFraction of WDIR=0 per station:")
print(weather_df.groupby('station_name').apply(
    lambda g: (g['Wind Direction'] == 0).mean()
))

print("\nWDIR=0 with WSPD>0 (the broken-sensor signal) per station:")
print(weather_df.groupby('station_name').apply(
    lambda g: ((g['Wind Direction'] == 0) & (g['Wind Speed'] > 0)).sum()
))

# 3. Is Maximum Wind Speed still polluted?
print("\nMaximum Wind Speed:")
print(weather_df['Maximum Wind Speed'].describe())
print(f"Rows with impossibly high MaxWS (>100): {(weather_df['Maximum Wind Speed'] > 100).sum()}")

# 4. Wind Speed sentinels — already mostly cleaned but double-check
print(f"\nRows with WSPD > 50 m/s: {(weather_df['Wind Speed'] > 50).sum()}")
print(f"Rows with WSPD == 999.9: {(weather_df['Wind Speed'] == 999.9).sum()}")

Station coverage:
                                                  min  \
station_name                                            
63rd Street Weather Station 2015-04-25 09:00:00-05:00   
Foster Weather Station      2015-05-22 16:00:00-05:00   
Oak Street Weather Station  2015-05-22 15:00:00-05:00   

                                                  max  count  
station_name                                                  
63rd Street Weather Station 2025-03-18 00:30:00-05:00  49948  
Foster Weather Station      2026-04-23 11:00:00-05:00  79169  
Oak Street Weather Station  2026-04-23 11:00:00-05:00  73566  

WDIR stats per station:
                             nunique        mean         std
station_name                                                
63rd Street Weather Station      360  175.481192  102.661300
Foster Weather Station           360   91.773764  119.546955
Oak Street Weather Station       360  166.283242  121.734975

Fraction of WDIR=0 per station:
station_name
63rd 

/var/folders/9g/gwn33xx13pd4b2km54vx15qw0000gn/T/ipykernel_5222/2682100050.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(weather_df.groupby('station_name').apply(
/var/folders/9g/gwn33xx13pd4b2km54vx15qw0000gn/T/ipykernel_5222/2682100050.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(weather_df.groupby('station_name').apply(


In [20]:


def clean_weather(df):
    """Clean COC weather station data: drop pre-2021, fix sentinels, mark Foster's WDIR unusable."""
    df = df.copy()
    
    # 1. Filter to 2021+ (we don't model earlier readings)
    df = df[df['timestamp_central'].dt.year >= 2021].copy()
    
    # 2. Foster's wind direction is broken (53% bad). Set to NaN entirely.
    foster_mask = df['station_name'] == 'Foster Weather Station'
    df.loc[foster_mask, 'Wind Direction'] = np.nan
    
    # 3. For other stations, only the WDIR=0 & WSPD>0 rows are bad
    bad_wdir = (df['Wind Direction'] == 0) & (df['Wind Speed'] > 0) & ~foster_mask
    df.loc[bad_wdir, 'Wind Direction'] = np.nan
    
    # 4. Maximum Wind Speed has 1 row with sensor glitch (445.2). Anything above 50 m/s is fake.
    df.loc[df['Maximum Wind Speed'] > 50, 'Maximum Wind Speed'] = np.nan
    
    return df

weather_clean = clean_weather(weather_df)

# Verify
print(f"Shape: {weather_clean.shape} (was {weather_df.shape})")
print(f"\nNaN counts per station for key columns:")
for station in weather_clean['station_name'].unique():
    sub = weather_clean[weather_clean['station_name'] == station]
    print(f"\n  {station} ({len(sub)} rows):")
    print(f"    WSPD NaN:     {sub['Wind Speed'].isna().sum()}")
    print(f"    WDIR NaN:     {sub['Wind Direction'].isna().sum()}")
    print(f"    MaxWS NaN:    {sub['Maximum Wind Speed'].isna().sum()}")
    print(f"    WDIR range:   {sub['Wind Direction'].min()} to {sub['Wind Direction'].max()}")
print(f"\nMaximum Wind Speed range: {weather_clean['Maximum Wind Speed'].min()} to {weather_clean['Maximum Wind Speed'].max()}")

Shape: (92706, 19) (was (202698, 19))

NaN counts per station for key columns:

  Foster Weather Station (41620 rows):
    WSPD NaN:     0
    WDIR NaN:     41620
    MaxWS NaN:    0
    WDIR range:   nan to nan

  Oak Street Weather Station (40476 rows):
    WSPD NaN:     2
    WDIR NaN:     927
    MaxWS NaN:    2
    WDIR range:   0.0 to 359.0

  63rd Street Weather Station (10610 rows):
    WSPD NaN:     0
    WDIR NaN:     10
    MaxWS NaN:    0
    WDIR range:   0.0 to 359.0

Maximum Wind Speed range: 0.0 to 25.5


In [21]:
weather_clean.head(2)

,station_name,Measurement Timestamp,Air Temperature,Wet Bulb Temperature,Humidity,Rain Intensity,COC_mmprecip_lag1h,Total Rain,Precipitation Type,Wind Direction,Wind Speed,Maximum Wind Speed,Barometric Pressure,Solar Radiation,Heading,Battery Life,Measurement Timestamp Label,Measurement ID,timestamp_central
0,Foster Weather Station,04/23/2026 11:00:00 AM,16.5,NaN,75,NaN,0.0,NaN,NaN,NaN,3.3,0.0,989.8,685,NaN,15.1,04/23/2026 11:00 AM,FosterWeatherStation202604231100,2026-04-23 11:00:00-05:00
1,Oak Street Weather Station,04/23/2026 11:00:00 AM,21.0,16.9,67,0.0,0.0,101.8,0.0,110.0,4.1,5.1,991.5,689,358.0,12.0,04/23/2026 11:00 AM,OakStreetWeatherStation202604231100,2026-04-23 11:00:00-05:00


In [22]:
wind_weather_clean = weather_clean.copy()
wind_weather_clean['timestamp_central'] = (
    pd.to_datetime(
        weather_df['Measurement Timestamp'],
        format='%m/%d/%Y %I:%M:%S %p',
        errors='coerce'
    )
    .dt.tz_localize('America/Chicago', nonexistent='NaT', ambiguous='NaT')) #nonexistent='shift_forward' handles the spring-forward case (the hour that doesn't exist gets bumped up). For fall-back, ambiguous='infer' works if your data is monotonic; 

#
#making a functional df_for eda
wind_weather_clean = wind_weather_clean.rename(columns = {'Wind Speed':'WSPD', 'Maximum Wind Speed':'GST', 'Wind Direction':'WDIR', 'Station Name' : 'station_name'})
# wind_weather_clean = wind_weather_clean[['station_name', 'WSPD','GST','WDIR','timestamp_central']]



In [23]:

wind_weather_clean


,station_name,Measurement Timestamp,Air Temperature,Wet Bulb Temperature,Humidity,Rain Intensity,COC_mmprecip_lag1h,Total Rain,Precipitation Type,WDIR,WSPD,GST,Barometric Pressure,Solar Radiation,Heading,Battery Life,Measurement Timestamp Label,Measurement ID,timestamp_central
0,Foster Weather Station,04/23/2026 11:00:00 AM,16.50,NaN,75,NaN,0.0,NaN,NaN,NaN,3.3,0.0,989.8,685,NaN,15.1,04/23/2026 11:00 AM,FosterWeatherStation202604231100,2026-04-23 11:00:00-05:00
1,Oak Street Weather Station,04/23/2026 11:00:00 AM,21.00,16.9,67,0.0,0.0,101.8,0.0,110.0,4.1,5.1,991.5,689,358.0,12.0,04/23/2026 11:00 AM,OakStreetWeatherStation202604231100,2026-04-23 11:00:00-05:00
2,Foster Weather Station,04/23/2026 10:00:00 AM,16.28,NaN,75,NaN,0.0,NaN,NaN,NaN,3.3,0.0,990.2,568,NaN,15.1,04/23/2026 10:00 AM,FosterWeatherStation202604231000,2026-04-23 10:00:00-05:00
3,Oak Street Weather Station,04/23/2026 10:00:00 AM,21.40,17.0,64,0.0,0.0,101.8,0.0,121.0,2.4,3.5,991.9,590,358.0,12.0,04/23/2026 10:00 AM,OakStreetWeatherStation202604231000,2026-04-23 10:00:00-05:00
4,Foster Weather Station,04/23/2026 09:00:00 AM,19.67,NaN,65,NaN,0.0,NaN,NaN,NaN,3.3,0.0,990.5,423,NaN,15.1,04/23/2026 9:00 AM,FosterWeatherStation202604230900,2026-04-23 09:00:00-05:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92711,Foster Weather Station,05/17/2021 04:00:00 PM,10.28,NaN,71,NaN,0.0,NaN,NaN,NaN,3.3,0.0,1001.0,323,NaN,15.2,05/17/2021 4:00 PM,FosterWeatherStation202105171600,2021-05-17 16:00:00-05:00
92712,Oak Street Weather Station,05/17/2021 04:00:00 PM,16.10,13.6,77,0.0,0.0,18.6,0.0,60.0,1.1,1.5,1001.2,255,357.0,12.0,05/17/2021 4:00 PM,OakStreetWeatherStation202105171600,2021-05-17 16:00:00-05:00
92713,63rd Street Weather Station,05/17/2021 03:00:00 PM,15.70,14.0,83,0.0,0.0,11.0,0.0,51.0,1.3,2.7,1000.6,467,352.0,11.8,05/17/2021 3:00 PM,63rdStreetWeatherStation202105171500,2021-05-17 15:00:00-05:00
92714,Foster Weather Station,05/17/2021 03:00:00 PM,8.39,NaN,71,NaN,0.0,NaN,NaN,NaN,3.3,0.0,1000.7,321,NaN,15.1,05/17/2021 3:00 PM,FosterWeatherStation202105171500,2021-05-17 15:00:00-05:00


In [26]:
station_location

# #join before making the station name change
station_location_2 = station_location.rename(columns = {'Sensor Name':'station_name'}).copy()

# print(station_location.head(10))
COC_wind_weather_wlatlon = duckdb.query("""
SELECT
    b.station_name, 
    b.WSPD, 
    b.GST,
    b.WDIR,
    b.timestamp_central,
    s.Latitude AS station_lat,
    s.Longitude AS station_lon,
    FROM wind_weather_clean b   
    JOIN station_location_2 s ON b.station_name = s.station_name       
""").df()


In [35]:
COC_wind_weather_wlatlon.dtypes

station_name                                  object
WSPD                                         float64
GST                                          float64
WDIR                                         float64
timestamp_central    datetime64[us, America/Chicago]
station_lat                                  float64
station_lon                                  float64
dtype: object

In [32]:
COC_wind_weather_final = COC_wind_weather_wlatlon.copy()
COC_wind_weather_final['station_name'] = COC_wind_weather_final['station_name'].str.replace(" ","_").str.lower().str.replace("_weather_station","").copy()

In [ ]:
COC_wind_weather_final
# commenting out to not overwrite!
# COC_wind_weather_final.to_parquet("data/prepared_data/COC_wind.parquet", index=False)